[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/byu-matrix-lab/torchlingo/blob/main/docs/docs/course/lecture-11-comet-on-a6.ipynb)


# Lecture 11: COMET on your A6 sentences, against BLEU, chrF and your own ranking

In A6 you scored two MT systems with BLEU and chrF and ranked 50 of their translations by hand.
This adds two learned metrics, COMET with a reference and COMETKiwi without one, the two you ran
in A10, and asks the question A11 is about: do the metrics agree with each other, and with you?
Then it asks the same of your own A8 and A9 systems.

By the end you will be able to:

1. score both systems with COMET and COMETKiwi on the same sentences as BLEU and chrF;
2. measure how often two metrics, or a metric and you, prefer the same system, sentence by
   sentence;
3. say whether each COMET model agrees with BLEU, chrF and your ranking, and why;
4. say what an LLM-as-judge scores, and why agreeing on the better system is not agreeing on
   each sentence;
5. say whether COMET and COMETKiwi agree with BLEU that your A9 model's pieces helped.

These serve Lecture 11's outcome *Run COMET and COMETKiwi on their own A6 data and on their own
A8 and A9 test output; reconcile them with BLEU, chrF and their human ranking; state what
LLM-as-judge scores and its system- against segment-level caveat*.

You write two functions, marked TODO, and one prediction; checks tell you when each is done, and
the notebook runs the rest. **Running in Google Colab?** Runtime → Change runtime type → any GPU
(a T4 is fine). COMETKiwi needs your Hugging Face setup from A10: before Run all, click the key
icon in the left sidebar and turn on **Notebook access** for `HF_TOKEN` in this notebook. Then
Run all; it stops at each TODO until you write it, and at the prediction until you write one.
The scores are saved to Drive as soon as they are computed, so a dropped session never costs you
the scoring: run Step 1 and the cells above it, then Step 3, then Step 5; Step 6 says the same
for its own scores. **When you finish, disconnect the GPU**: Runtime → Disconnect and delete
runtime. Not while a scoring cell runs, which would stop it. The last cell says why.

<!-- GENERATED:purpose - change the torchlingo metadata, not this cell -->

**CS 479 course notebook** — homework for Lecture 11 (*LLM Judges, Sentences and Spans*). A head start on **A11**.

In [ ]:
# Install TorchLingo and COMET in Colab. Nothing to uncomment. Pip's "you may need to restart
# the kernel" does not apply to a first install; if the install fails, the import below stops here.
import sys
if "google.colab" in sys.modules:
    %pip install --quiet "torchlingo @ git+https://github.com/byu-matrix-lab/torchlingo@course"
    # pip keeps a TorchLingo already in this session when its version number matches, even if
    # it is older code; this swaps in the course branch's, leaving the dependencies alone.
    %pip install --quiet --force-reinstall --no-deps "torchlingo @ git+https://github.com/byu-matrix-lab/torchlingo@course"
    # COMET asks for numpy older than 2, which would break packages Colab ships built for
    # numpy 2; COMET itself runs on numpy 2. So its other requirements go in first, then COMET.
    # pip may then print a red list of conflicts with other packages Colab ships; this uses none.
    %pip install --quiet "entmax>=1.1,<2" "jsonargparse==3.13.1" "pytorch-lightning>=2,<3" "sacrebleu>=2,<3" "setuptools<81" "torchmetrics>=0.10.2,<0.11" "transformers>=4.17,<5"
    %pip install --quiet --no-deps unbabel-comet
    from importlib.metadata import version
    if int(version("numpy").split(".")[0]) < 2:
        raise RuntimeError("An earlier COMET install replaced numpy in this session, which breaks "
                           "COMET. Runtime -> Disconnect and delete runtime, then Run all again.")
import torchlingo


In [ ]:
# Check the setup and connect Google Drive, where your A6 files are.
from torchlingo.colab import course_dir, setup

env = setup(drive=True)
A6_DIR = course_dir(env.drive_dir, "assignment6")   # MyDrive/CS479/assignment6


## Step 1: your A6 sentences and your MTEval ranks

Put your A6 files in `MyDrive/CS479/assignment6` as plain text, one sentence per line, line *n*
of each belonging to source sentence *n*: the sources, each system's translations, and the
references. If your A6 data is in another shape, such as one spreadsheet with a column for each,
save each column as its own `.txt` file first. Set the file names below.

Put your **MTEval results file** from A6 there too, and set its name. It has a header row, the
word `source` and then your two systems' names, and one row for each sentence you ranked: the
source, then each system's rank, where 1 is the better of the two and equal ranks are a tie.
**Use the two names in its header as the keys of `SYSTEM_FILES`**, exactly as written there.

The cell after that reads your ranks and finds each ranked sentence in your source file,
ignoring differences in spacing. It says how many it found, lists any it did not, and computes
each system's average rank, the number you reported in A6: a system you preferred every time
averages 1.0, and closer to 1 is better.

In [ ]:
SOURCE_FILE = "source.txt"          # <- your file names, in MyDrive/CS479/assignment6
REFERENCE_FILE = "reference.txt"
SYSTEM_FILES = {"system 1": "system1.txt", "system 2": "system2.txt"}   # <- your MTEval header's names
MTEVAL_FILE = "mteval_results.tsv"  # <- your MTEval results file from A6

# Stops, naming every file it cannot find, before reading any of them.
course_dir(
    env.drive_dir, "assignment6",
    needs=[SOURCE_FILE, REFERENCE_FILE, *SYSTEM_FILES.values(), MTEVAL_FILE],
    hint="Put your A6 files there, or fix the names above.",
)

In [ ]:
# Setup: run this cell, no need to read it. It reads your A6 files and your MTEval ranks.


def read_lines(name):
    """The file's lines. utf-8-sig drops a byte-order mark; reading as text turns Windows line
    endings into "\n"; splitting on "\n" alone keeps other line-break characters inside a line."""
    return (A6_DIR / name).read_text(encoding="utf-8-sig").rstrip("\n").split("\n")


def same_text(sentence):
    """The sentence with its spacing normalized, to match MTEval's rows to your sources."""
    return " ".join(sentence.split())


def read_mteval(name, systems):
    """Return the ranked sources, and {system: [rank, ...]}, from MTEval's results file."""
    lines = read_lines(name)
    header = lines[0].split("\t")
    if len(header) != 3 or set(header[1:]) != set(systems):
        raise SystemExit(
            f"{name}'s header is {header}: a source column, then your two systems' names. "
            f"SYSTEM_FILES names {list(systems)}; use the header's two names as its keys."
        )
    ranked, ranks = [], {system: [] for system in header[1:]}
    for n, line in enumerate(lines[1:], start=2):
        cells = line.split("\t")
        if len(cells) != 3 or not all(cell.strip().isdigit() for cell in cells[1:]):
            raise SystemExit(f"{name}, line {n}: expected a sentence and two ranks, got {cells}")
        ranked.append(cells[0])
        for system, cell in zip(header[1:], cells[1:]):
            ranks[system].append(int(cell))
    return ranked, ranks


NOTES = A6_DIR / "a11-notes.json"   # the scoring cell saves the scores here; the report reads them
sources, references = read_lines(SOURCE_FILE), read_lines(REFERENCE_FILE)
translations = {name: read_lines(f) for name, f in SYSTEM_FILES.items()}
print(f"{len(sources):,} source sentences; systems: {', '.join(SYSTEM_FILES)}.")

ranked, ranks = read_mteval(MTEVAL_FILE, SYSTEM_FILES)
line_of = {}
for i, sentence in enumerate(sources):
    line_of.setdefault(same_text(sentence), i)
matched = [line_of.get(same_text(sentence)) for sentence in ranked]   # a line of your files, or None
found = sum(i is not None for i in matched)
print(f"{MTEVAL_FILE}: {len(ranked)} sentences you ranked; {found} found in {SOURCE_FILE}.")
for sentence, i in zip(ranked, matched):
    if i is None:
        print(f"  not found: {sentence}")
if not found:
    raise SystemExit(f"None of the sentences you ranked is in {SOURCE_FILE}. Is it the source "
                     "file you gave MTEval?")

HUMAN_AVERAGE_RANK = {system: sum(r) / len(r) for system, r in ranks.items()}
ties = sum(r1 == r2 for r1, r2 in zip(*ranks.values()))
print("Your average ranks, as in A6: "
      + ", ".join(f"{system} {rank:.2f}" for system, rank in HUMAN_AVERAGE_RANK.items())
      + f"; {ties} of {len(ranked)} ranked as a tie.")

## Step 2: `comet_inputs`, from A10

**Yours: the function you wrote in A10**, which turns aligned sentences into the records COMET
scores and refuses lists of different lengths. Paste it in place of the TODO; the same checks
run.


In [ ]:
def comet_inputs(sources, translations, references=None):
    """Return one COMET record per sentence: src, mt, and ref when references are given.

    TODO: paste your A10 function. It raises ValueError if translations, or references when
    given, are not the same length as sources.
    """
    raise NotImplementedError("paste comet_inputs from A10, then run this cell again")


got = comet_inputs(["hola"], ["hello"], ["hi"])
assert got == [{"src": "hola", "mt": "hello", "ref": "hi"}], (
    f"with a reference, each record is {{'src': ..., 'mt': ..., 'ref': ...}}; got {got}")
got = comet_inputs(["hola"], ["hello"])
assert got == [{"src": "hola", "mt": "hello"}], (
    f"without a reference, each record has only 'src' and 'mt'; got {got}")
for bad in ((["a", "b"], ["x"], None), (["a"], ["x"], ["r", "s"])):
    try:
        got = comet_inputs(*bad)
    except ValueError:
        continue
    raise AssertionError(f"comet_inputs must raise ValueError for lists of different lengths, "
                         f"{bad}; got {got}")
for name, mt in translations.items():
    comet_inputs(sources, mt, references)   # your files are aligned, or this says which is not
print("comet_inputs passes its checks, and your files are aligned.")

## Step 3: yours to write, `pairwise_agreement`

With two systems, "do two metrics correlate?" has a sentence-by-sentence answer: for each
sentence, does each metric prefer the same system? Step 5 also uses it on your MTEval ranks,
which is how it measures how often a metric agrees with you.

`pairwise_agreement(a_1, a_2, b_1, b_2)` takes two metrics' scores, metric a's for systems 1 and
2 and metric b's for the same two, one score per sentence, and returns `(agree, counted)`:

- Higher is better for every score here. For sentence *i*, metric a prefers system 1 if
  `a_1[i] > a_2[i]`, system 2 if `a_1[i] < a_2[i]`, and **neither if they are equal**: a tie is
  no preference. Likewise metric b.
- `counted` is the number of sentences where **both** metrics prefer a system.
- `agree` is how many of those they prefer the same one on.

In [ ]:
def pairwise_agreement(a_1, a_2, b_1, b_2):
    """Return (agree, counted) for metric a's and metric b's scores of systems 1 and 2.

    TODO: write it, as Step 3 above specifies.
    """
    raise NotImplementedError("write pairwise_agreement, then run this cell again")


# A case you can check by hand: agree, disagree, a tie in a, a tie in b, agree.
got = pairwise_agreement([2, 1, 5, 3, 0.9], [1, 2, 5, 1, 0.1],
                         [9, 9, 1, 4, 0.7], [1, 1, 2, 4, 0.2])
assert got == (2, 3), (
    f"pairwise_agreement is not right yet: here it should count 2 agreements out of 3, (2, 3); "
    f"got {got}")
print("pairwise_agreement passes its check.")

## Step 4: predict, then score

**Write a prediction** in `PREDICTION`: will COMET prefer the same system as BLEU, chrF and your
ranking, and will COMETKiwi agree with COMET? The cell stops until you write one: a prediction
made after seeing the scores teaches nothing.


In [ ]:
PREDICTION = ""   # <- your prediction, in a sentence or two

assert PREDICTION.strip(), "Write your prediction in PREDICTION, then run this cell again."
print("Prediction recorded.")


Now the next two cells log in to Hugging Face and score both systems with all four metrics, on
every sentence. On a GPU the scoring takes a few minutes for 500 sentences. **Leave the runtime
connected while it runs**: disconnecting stops it, and it starts again from the beginning. When it
ends, it saves the scores to `a11-notes.json` beside your A6 files on Drive.

**COMETKiwi is the estimator you ran in A10.** It scores a translation from the source alone,
with no reference: that is estimation, where COMET with a reference is evaluation. It downloads
only for a Hugging Face account that has accepted its license, so it needs your A10 setup: the
license accepted, and your token saved in Colab as the secret `HF_TOKEN`. **Notebook access is
set per notebook**: click the key icon in the left sidebar and turn it on for `HF_TOKEN` here,
even though it was on in A10. COMETKiwi's scores run from 0 to 1, like COMET's, but the two are
still not comparable: each model has its own scale, so compare which system each prefers, never
one model's number with the other's.

In [ ]:
from google.colab import userdata
from huggingface_hub import login

HF_TOKEN = userdata.get("HF_TOKEN")   # the secret's name from A10, if you used a different one
if not HF_TOKEN:
    raise RuntimeError(
        "No HF_TOKEN found in Colab Secrets, so COMETKiwi cannot download. Click the key icon in "
        "the left sidebar and turn on 'Notebook access' for HF_TOKEN in this notebook (it is set "
        "per notebook, so A10's does not carry over); without the secret, add it as A10 says. "
        "Then run this cell again."
    )
login(token=HF_TOKEN)
print("Logged in to Hugging Face.")

In [ ]:
import json

import torch
from comet import download_model, load_from_checkpoint
from sacrebleu.metrics import CHRF
from torchlingo.evaluation import compute_bleu, compute_chrf

GPUS = 1 if torch.cuda.is_available() else 0
sentence_chrf = CHRF(word_order=2)          # the chrF++ that compute_chrf uses

# Each metric's score for each system, overall and per sentence: system_scores["COMET"]["DeepL"].
system_scores, sentence_scores = {}, {}
for model_name, label, with_refs in (("Unbabel/wmt22-comet-da", "COMET", True),
                                     ("Unbabel/wmt22-cometkiwi-da", "COMETKiwi", False)):
    # COMET reports any failed download as "not supported by COMET", which hides the usual cause.
    try:
        model = load_from_checkpoint(download_model(model_name))
    except KeyError as error:
        raise RuntimeError(
            f"{label} did not download. Logged in, the usual cause is that this Hugging Face "
            f"account has not accepted {label}'s license: accept it at "
            f"https://huggingface.co/{model_name} with the account the token belongs to, then "
            "run this cell again."
        ) from error
    system_scores[label], sentence_scores[label] = {}, {}
    for name, mt in translations.items():
        out = model.predict(comet_inputs(sources, mt, references if with_refs else None),
                            batch_size=32, gpus=GPUS, num_workers=2, progress_bar=False)
        system_scores[label][name] = float(out.system_score)
        sentence_scores[label][name] = [float(s) for s in out.scores]

system_scores["BLEU"], system_scores["chrF"], sentence_scores["chrF"] = {}, {}, {}
for name, mt in translations.items():
    system_scores["BLEU"][name] = compute_bleu(mt, references).score
    system_scores["chrF"][name] = compute_chrf(mt, references).score
    sentence_scores["chrF"][name] = [sentence_chrf.sentence_score(h, [r]).score
                                     for h, r in zip(mt, references)]

# Saved to Drive, so the report survives a dropped session without scoring again.
NOTES.write_text(json.dumps({"prediction": PREDICTION.strip(), "system_scores": system_scores,
                             "sentence_scores": sentence_scores}), encoding="utf-8")
print(f"Scored with COMET, COMETKiwi, BLEU and chrF; saved to {NOTES.name} on Drive.")

## Step 5: the report

The system scores, which system each metric prefers, and, sentence by sentence, how often the
metrics agree with each other and with you. With two systems, "correlates with your ranking" at
the system level means "prefers the same system"; the sentence-level counts say how firmly.
Against you, the counts use your own `pairwise_agreement` on the sentences you ranked, so a
sentence you ranked as a tie expresses no preference and is left out.

The report reads the scores from `a11-notes.json` on Drive. If Colab dropped your session after
scoring, you need not score again: run Step 1 and the cells above it, then Step 3, then this
cell.

In [ ]:
import json

notes = json.loads(NOTES.read_text(encoding="utf-8"))   # saved by the scoring cell
system_scores, sentence_scores = notes["system_scores"], notes["sentence_scores"]
if "COMETKiwi" not in system_scores:
    raise SystemExit(f"{NOTES.name} was saved before A11 switched to COMETKiwi, so it holds "
                     "another model's scores. Run Step 4 again: the login, then the scoring cell.")

one, two = SYSTEM_FILES
print("=== For the A11 summary ===")
print(f"{len(sources):,} sentences scored automatically; systems: {one} and {two}\n")
print(f"{'metric':12} {one:>12} {two:>12}   prefers")
for label in ("BLEU", "chrF", "COMET", "COMETKiwi"):
    s1, s2 = system_scores[label][one], system_scores[label][two]
    print(f"{label:12} {s1:12.3f} {s2:12.3f}   {one if s1 > s2 else two}")
h1, h2 = HUMAN_AVERAGE_RANK[one], HUMAN_AVERAGE_RANK[two]
prefers = one if h1 < h2 else two if h2 < h1 else "neither"
print(f"{'human rank':12} {h1:12.2f} {h2:12.2f}   {prefers}  (lower is better; all {len(ranked)} you ranked)")

print("\nSentence by sentence, how often two metrics prefer the same system:")
for a, b in (("COMET", "chrF"), ("COMETKiwi", "chrF"), ("COMETKiwi", "COMET")):
    agree, counted = pairwise_agreement(sentence_scores[a][one], sentence_scores[a][two],
                                        sentence_scores[b][one], sentence_scores[b][two])
    print(f"  {a} and {b}: {agree} of {counted} ({100 * agree / max(counted, 1):.0f}%)")

# Your ranks as scores where higher is better, like every score above: minus the rank. A
# sentence you ranked as a tie is then no preference, and pairwise_agreement leaves it out.
rows = [(k, i) for k, i in enumerate(matched) if i is not None]
you_1, you_2 = [-ranks[one][k] for k, _ in rows], [-ranks[two][k] for k, _ in rows]
ties = sum(r1 == r2 for r1, r2 in zip(you_1, you_2))
print(f"\nOn the {len(rows)} sentences you ranked and Step 1 found, how often each metric prefers "
      f"the system you did; {ties} you ranked as a tie are not counted:")
for label in ("chrF", "COMET", "COMETKiwi"):
    agree, counted = pairwise_agreement([sentence_scores[label][one][i] for _, i in rows],
                                        [sentence_scores[label][two][i] for _, i in rows],
                                        you_1, you_2)
    print(f"  {label} and you: {agree} of {counted} ({100 * agree / max(counted, 1):.0f}%)")
print(f"\nYour prediction: {notes['prediction']}")

## Step 6: your own systems

Steps 1 to 5 turn the metrics on two commercial systems. This step turns them on **yours**: your
A8 model (words) and your A9 model (pieces), on your own A8 test set. A9 reported, in BLEU,
whether the pieces helped. Do COMET and COMETKiwi agree?

The cell reads, from `MyDrive/CS479/assignment8`, where A8 and A9 left them: `test.tsv`, your
test set; `test-a9.hyp`, your A9 model's translations of it; and your A8 model's translations,
from `test-a8-again.hyp` if your A9 notebook saved one (its Part B translates your A8 model again,
so both are scored the way A9 reported them), and otherwise from `test.hyp`, which the A8 notebook
wrote. It says which it used. There is nothing to set. BLEU is computed as A9 computed it,
spBLEU if your language is written without spaces.

It needs the Hugging Face login above, and it loads both COMET models itself: two systems, two
models, on about 2,000 sentences, a few minutes on a T4. **Leave the runtime connected while it
runs.** The scores go to `a11-own-notes.json` in `MyDrive/CS479/assignment11`; your A6 scores in
`a11-notes.json` are not touched. If you finished Steps 1 to 5 in an earlier session, run the two
cells at the top of the notebook, Steps 1 to 3, and the login cell, then this step.

**Does COMET read your language?** Both models read through a multilingual encoder, XLM-R for
COMET and InfoXLM for COMETKiwi, trained on about 100 languages:
[the list](https://github.com/Unbabel/COMET#languages-covered). If your language is not among
them, the scores mean less, because the model is reading text it was not trained on; check, and
if so, say so in your answer to question 5.

In [ ]:
# Setup: run this cell, no need to read it. It reads your A8 and A9 translations of your test
# set, scores both systems with all four metrics, and saves the scores to Drive.
import json

import pandas as pd
import sacrebleu
import torch
from comet import download_model, load_from_checkpoint
from sacrebleu.metrics import CHRF
from torchlingo.evaluation import compute_chrf

A8_DIR = course_dir(
    env.drive_dir, "assignment8", needs=["test.tsv", "test-a9.hyp"],
    hint="test.tsv is your test set, written by the A8 kickoff notebook; test-a9.hyp is your A9 "
         "model's translations of it, written by Part B of the A9 notebook.",
)
# A8 as A9 scored it, if A9 saved its own translation of your A8 model; else A8's own test.hyp.
A8_HYP = next((f for f in ("test-a8-again.hyp", "test.hyp") if (A8_DIR / f).exists()), None)
if A8_HYP is None:
    raise SystemExit(f"No translations by your A8 model in {A8_DIR}: test.hyp is written by the "
                     "A8 kickoff notebook's last step, test-a8-again.hyp by A9's Part B.")

own_test = pd.read_csv(A8_DIR / "test.tsv", sep="\t", dtype=str, keep_default_na=False)
own_sources, own_references = list(own_test.src), list(own_test.tgt)


def read_hyp(name):
    """One translation per line, as A8 and A9 wrote them; an empty translation is still a line."""
    text = (A8_DIR / name).read_text(encoding="utf-8")
    return text[:-1].split("\n") if text.endswith("\n") else text.split("\n")


own_files = {"A8": A8_HYP, "A9": "test-a9.hyp"}
own_hyps = {system: read_hyp(name) for system, name in own_files.items()}
for system, name in own_files.items():
    if len(own_hyps[system]) != len(own_sources):
        raise SystemExit(f"{name} has {len(own_hyps[system]):,} lines and test.tsv has "
                         f"{len(own_sources):,} sentences, so they do not line up. Translate "
                         f"the test set again with your {system} model, then run this cell again.")
if A8_HYP == "test.hyp":
    print("A8: test.hyp, from the A8 notebook. Your A9 notebook translated your A8 model again "
          "without saving it, so if A8's BLEU below differs from the one A9 printed, say so.")

# BLEU as A9 scored it: spBLEU for a language written without spaces, SacreBLEU's default otherwise.
settings_file = A8_DIR / "a9-settings.json"
if settings_file.exists():
    NO_SPACES = json.loads(settings_file.read_text(encoding="utf-8"))["no_spaces"]
else:   # A9's test: a language without spaces averages under three "words" a line
    NO_SPACES = bool(own_test.tgt.str.split().str.len().mean() < 3)
TOKENIZE = "flores200" if NO_SPACES else "13a"

GPUS = 1 if torch.cuda.is_available() else 0
sentence_chrf = CHRF(word_order=2)          # the chrF++ that compute_chrf uses
own_system, own_sentence = {}, {}
for model_name, label, with_refs in (("Unbabel/wmt22-comet-da", "COMET", True),
                                     ("Unbabel/wmt22-cometkiwi-da", "COMETKiwi", False)):
    try:
        model = load_from_checkpoint(download_model(model_name))
    except KeyError as error:
        raise RuntimeError(
            f"{label} did not download. Logged in, the usual cause is that this Hugging Face "
            f"account has not accepted {label}'s license: accept it at "
            f"https://huggingface.co/{model_name} with the account the token belongs to, then "
            "run this cell again."
        ) from error
    own_system[label], own_sentence[label] = {}, {}
    for system, mt in own_hyps.items():
        out = model.predict(comet_inputs(own_sources, mt, own_references if with_refs else None),
                            batch_size=32, gpus=GPUS, num_workers=2, progress_bar=False)
        own_system[label][system] = float(out.system_score)
        own_sentence[label][system] = [float(s) for s in out.scores]

own_system["BLEU"], own_system["chrF"], own_sentence["chrF"] = {}, {}, {}
for system, mt in own_hyps.items():
    own_system["BLEU"][system] = sacrebleu.corpus_bleu(mt, [own_references], tokenize=TOKENIZE).score
    own_system["chrF"][system] = compute_chrf(mt, own_references).score
    own_sentence["chrF"][system] = [sentence_chrf.sentence_score(h, [r]).score
                                    for h, r in zip(mt, own_references)]

# Saved to its own file, so the A6 scores in a11-notes.json are never touched.
A11_DIR = course_dir(env.drive_dir, "assignment11", create=True)
(A11_DIR / "a11-own-notes.json").write_text(json.dumps({
    "files": own_files, "no_spaces": bool(NO_SPACES), "sources": own_sources,
    "references": own_references, "translations": own_hyps,
    "system_scores": own_system, "sentence_scores": own_sentence,
}, ensure_ascii=False), encoding="utf-8")
print(f"Scored A8 and A9 with COMET, COMETKiwi, BLEU and chrF on {len(own_sources):,} test "
      f"sentences; saved to a11-own-notes.json in {A11_DIR}.")

The report below reads `a11-own-notes.json`: the table, which system each metric prefers, the
sentence-by-sentence counts from your `pairwise_agreement`, and the first three sentences where
COMET and chrF disagree, each with both systems' scores. If Colab dropped your session after
scoring, run the two cells at the top of the notebook, then Step 3, then this cell.

In [ ]:
import json

# Read from Drive, so this report runs again without scoring again.
OWN_NOTES = course_dir(
    env.drive_dir, "assignment11", needs=["a11-own-notes.json"],
    hint="The cell above writes it when it scores your A8 and A9 systems; run that first.",
) / "a11-own-notes.json"
own = json.loads(OWN_NOTES.read_text(encoding="utf-8"))
scores, per_sentence, hyps = own["system_scores"], own["sentence_scores"], own["translations"]
which = "spBLEU (flores200)" if own["no_spaces"] else "SacreBLEU, default tokenizer"

print("=== For the A11 summary: your own systems ===")
print(f"{len(own['sources']):,} test sentences; A8 from {own['files']['A8']}, A9 from "
      f"{own['files']['A9']}; BLEU is {which}, as A9 scored it\n")
print(f"{'metric':12} {'A8':>10} {'A9':>10}   prefers")
for label in ("BLEU", "chrF", "COMET", "COMETKiwi"):
    s8, s9 = scores[label]["A8"], scores[label]["A9"]
    print(f"{label:12} {s8:10.3f} {s9:10.3f}   {'A9' if s9 > s8 else 'A8' if s8 > s9 else 'neither'}")

print("\nSentence by sentence, how often two metrics prefer the same system:")
for a, b in (("COMET", "chrF"), ("COMETKiwi", "chrF"), ("COMETKiwi", "COMET")):
    agree, counted = pairwise_agreement(per_sentence[a]["A8"], per_sentence[a]["A9"],
                                        per_sentence[b]["A8"], per_sentence[b]["A9"])
    print(f"  {a} and {b}: {agree} of {counted} ({100 * agree / max(counted, 1):.0f}%)")

print("\nThe first three sentences where COMET and chrF prefer different systems:")
comet, chrf = per_sentence["COMET"], per_sentence["chrF"]
shown = 0
for i, (src, ref) in enumerate(zip(own["sources"], own["references"])):
    c8, c9, f8, f9 = comet["A8"][i], comet["A9"][i], chrf["A8"][i], chrf["A9"][i]
    if c8 != c9 and f8 != f9 and (c8 > c9) != (f8 > f9):
        print(f"\nSRC {src}\nREF {ref}")
        print(f"A8  {hyps['A8'][i]}\n    COMET {c8:.3f}   chrF {f8:.1f}")
        print(f"A9  {hyps['A9'][i]}\n    COMET {c9:.3f}   chrF {f9:.1f}")
        shown += 1
        if shown == 3:
            break
if not shown:
    print("  none: wherever both prefer a system, they prefer the same one.")

## Step 7: the summary

A11's one-page summary reports the numbers above and answers these five questions, each in a few
sentences:

1. **Does each COMET model agree with BLEU, chrF and your ranking, and why or why not?** Judge
   the reference-based COMET and COMETKiwi separately: their scores sit on different scales, so
   compare which system each prefers, never one model's number with the other's. Answer for the
   system each prefers, and for the sentences you ranked, one by one.
2. **Your prediction against the result.** Where were you right, where wrong, and what does the
   miss tell you?
3. **What the reference changes.** COMET and COMETKiwi read the same translations; one also
   reads your reference. Find a sentence where they disagree and say what the reference told
   COMET that the source alone did not.
4. **LLM-as-judge.** Suppose you asked a large language model to score each translation from 0 to
   100, given the source. Is that evaluation or estimation? Such judges agree with people far
   better on which *system* is better than on which *sentence* is: use your own Step 5 counts,
   where a metric can prefer the same system you did while disagreeing with you on many of the
   sentences you ranked, to explain how that can be.
5. **Your own systems.** Do COMET and COMETKiwi agree with BLEU (as A9 reported it) about
   whether the pieces helped? Use the sentence-by-sentence count from Step 6: on how many
   sentences do COMET and chrF prefer the same system? Pick one sentence where they disagree and
   say which you believe, and why.

For the extra credit, xCOMET-XL is `Unbabel/XCOMET-XL` in the same loop. Like COMETKiwi, it
needs its license accepted on Hugging Face, and it needs a large GPU.

---
## When you finish: disconnect the GPU

**Runtime → Disconnect and delete runtime.** Colab charges compute units for every minute a GPU runtime stays connected, even when nothing is running, so a runtime left open after the work is done uses up your monthly units. Closing the browser tab does not end it right away. Your scores are saved on Drive, in `a11-notes.json` beside your A6 files and, for your own systems, in `a11-own-notes.json` in `MyDrive/CS479/assignment11`, so you can print either report again later without scoring again: Steps 5 and 6 each say which cells to run.